## Silver Layer: Aircraft

Cleans the raw OpenSky snapshot in `bronze_aircraft` and saves it to `silver_aircraft`,
which the FlightPulse API reads for live aircraft positions.

```text
workspace.default.bronze_aircraft  →  workspace.default.silver_aircraft
```

### What gets cleaned

| Step | Problem in the raw data | What Silver does |
|---|---|---|
| 1. Drop aircraft with no ID | A few records have no `icao24`, the aircraft's unique transponder code | Removes them; they can't be identified or tracked |
| 2. Drop aircraft with no position | OpenSky includes aircraft it can hear but not locate (no latitude/longitude), e.g. parked planes or older transponders | Removes them; they can't be placed on the map. This is why Silver has fewer aircraft than Bronze |
| 3. One row per aircraft | A snapshot can occasionally list the same aircraft twice | Keeps only its most recent record (latest `last_contact`) |
| 4. Readable times | Times are Unix seconds, e.g. `1790159810` | Adds `position_time` and `contact_time` as timestamps, e.g. `2026-09-23 10:36:50` |

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import *

bronze_df = spark.table("workspace.default.bronze_aircraft")

latest_first = Window.partitionBy("icao24").orderBy(desc("last_contact"))

silver_aircraft = bronze_df \
    .filter(col("icao24").isNotNull()) \
    .filter(col("longitude").isNotNull() & col("latitude").isNotNull()) \
    .withColumn("row", row_number().over(latest_first)) \
    .filter(col("row") == 1) \
    .withColumn("position_time", timestamp_seconds(col("time_position"))) \
    .withColumn("contact_time", timestamp_seconds(col("last_contact"))) \
    .select(
        "icao24",
        "callsign",
        "origin_country",
        "time_position",
        "longitude",
        "latitude",
        "baro_altitude",
        "on_ground",
        "velocity",
        "true_track",
        "vertical_rate",
        "sensors",
        "geo_altitude",
        "squawk",
        "spi",
        "position_source",
        "position_time",
        "contact_time"
    )

display(silver_aircraft.limit(10))

In [ ]:
silver_aircraft.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.silver_aircraft")

print("silver_aircraft rows:", spark.table("workspace.default.silver_aircraft").count())